# 00 · Download the raw data

**Goal:** get every source we chose in the data research onto this computer, exactly as published, before we clean or model anything.

**Scope:** New York City first, government sources, at least 5 years of history.

**What this notebook does:**
1. Reads the list of sources from `src/sources.py`. That is the only place URLs and versions live.
2. Downloads each one into `data/raw/<group>/<source>/`. Files already on disk are skipped, so it is safe to rerun.
3. Logs every file in `data/raw/manifest.csv`: URL, download time, size and SHA-256 fingerprint. That lets us prove later which file produced which number.
4. Dates the file name for snapshot sources, the ones whose publisher overwrites them (NYC Jobs, the NYS DOL wage table, and others). Each rerun keeps a new copy, so we build our own history.

**What it does not do:** clean, filter or reshape anything. That happens one source at a time in the next notebooks.

Background for every source: `docs/plans/data-research/README.md` and reports `00`–`06`.

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import RAW
import sources as S

TODAY = date.today().isoformat()
REGISTRY = S.build_registry()
session = requests.Session()
print(f"{len(REGISTRY)} files in the registry, saving under {RAW}")

155 files in the registry, saving under <repo>\data\raw


## The list of sources
One row per file. `snapshot` = the publisher overwrites it, so we keep dated copies.

In [2]:
reg = pd.DataFrame(REGISTRY)
reg["snapshot"] = reg.get("snapshot", False).fillna(False).astype(bool)
reg.groupby(["group", "key"]).agg(files=("file", "count"), snapshot=("snapshot", "max"),
                                  note=("note", "first")).reset_index()

,group,key,files,snapshot,note
0,bls_timeseries,bls_cu,7,True,CPI-U all items by area (NY-Newark-Jersey City...
1,bls_timeseries,bls_jt,11,True,"JOLTS openings/hires/quits/layoffs, US + state..."
2,bls_timeseries,bls_la,7,True,"LAUS unemployment, NY areas incl. the 5 boroug..."
3,graduates,cuny_degrees_by_type,1,False,CUNY degrees granted by type (tie-out)
4,graduates,cuny_enrollment,1,False,CUNY enrollment trends by college
5,graduates,cuny_programs,1,False,"CUNY graduates and fall enrollment by program,..."
6,graduates,ipeds_completions,10,False,"Completions by college x CIP x award, academic..."
7,graduates,ipeds_dictionary,1,False,NaN
8,graduates,ipeds_directory,10,False,Institution directory
9,graduates,nces_cip,3,False,CIP 2010 -> 2020 (IPEDS before C2020 uses CIP ...


## Download, group by group

Each group below is one part of the question chain:

| Group | Answers |
|---|---|
| `jobs` | How many people work in each job, pay, projected openings (2018–2025 and 4 projection editions); NYC economic output (GDP) by industry; which jobs each industry hires (NY State 2019–2024, US matrix) |
| `hiring` | Hires and separations by age group in the 5 boroughs; unemployment benefits |
| `licensing` | Licensed people per profession 1997–2025; nursing exam pass rates by school |
| `graduates` | CUNY and other NYC college graduates by major, 2015–2025; CUNY enrollment by program |
| `outcomes` | What graduates earn and do (PSEO, Scorecard, Census microdata, NY Fed) |
| `pay` | Entry-level wage levels by year; NYC government pay by year |
| `bls_timeseries` | NY metro prices (CPI), NY State openings/hires/layoffs (JOLTS), borough unemployment (LAUS) |
| `skills` / `postings` | Skills per job (O\*NET); current NYC government postings; Indeed postings index |

The biggest files are the Census microdata (~140 MB), the Census hiring files (~160 MB each) and the NYS DOL industry table. The first run takes a while.

In [3]:
def run_group(group):
    rows = [S.fetch(src, TODAY, session) for src in REGISTRY if src["group"] == group]
    out = pd.DataFrame(rows)
    out["MB"] = (out["bytes"] / 1e6).round(1)
    return out.drop(columns="bytes")

log = {}

### `jobs`

In [4]:
log["jobs"] = run_group("jobs")
log["jobs"]

,file,status,MB
0,jobs/bls_oews_metro/oesm18ma.zip,already have,31.2
1,jobs/bls_oews_metro/oesm19ma.zip,already have,35.5
2,jobs/bls_oews_metro/oesm20ma.zip,already have,36.4
3,jobs/bls_oews_metro/oesm21ma.zip,already have,38.9
4,jobs/bls_oews_metro/oesm22ma.zip,already have,39.0
5,jobs/bls_oews_metro/oesm23ma.zip,already have,39.2
6,jobs/bls_oews_metro/oesm24ma.zip,already have,40.2
7,jobs/bls_oews_metro/oesm25ma.zip,already have,39.9
8,jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv,already have,0.7
9,jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv,already have,0.7


### `hiring`

In [5]:
log["hiring"] = run_group("hiring")
log["hiring"]

,file,status,MB
0,hiring/census_qwi/qwi_ny_sa_f_gc_ns_op_u.csv.gz,already have,167.2
1,hiring/census_qwi/qwi_ny_se_f_gc_ns_op_u.csv.gz,already have,120.7
2,hiring/census_qwi/qwi_ny.md5sum,already have,0.0
3,hiring/census_qwi/version_qwi.txt,already have,0.0
4,hiring/nysdol_ui_beneficiaries/xbjp-8sra.csv,already have,0.8


### `licensing`

In [6]:
log["licensing"] = run_group("licensing")
log["licensing"]

,file,status,MB
0,licensing/nysed_registrants/numbers-registered...,already have,0.0
1,licensing/nysed_registrants/numbers-registered...,already have,0.0
2,licensing/nysed_registrants/numbers-registered...,already have,0.0
3,licensing/nysed_registrants/numbers-registered...,already have,0.0
4,licensing/nysed_registrants/numbers-registered...,already have,0.0
5,licensing/nysed_registrants/numbers-registered...,already have,0.0
6,licensing/nysed_registrants/numbers-registered...,already have,0.0
7,licensing/nysed_registrants/numbers-registered...,already have,0.0
8,licensing/nysed_registrants/numbers-registered...,already have,0.0
9,licensing/nysed_registrants/numbers-registered...,already have,0.0


### `graduates`

In [7]:
log["graduates"] = run_group("graduates")
log["graduates"]

,file,status,MB
0,graduates/ipeds_completions/C2015_A.zip,already have,8.9
1,graduates/ipeds_directory/HD2015.zip,already have,1.1
2,graduates/ipeds_completions/C2016_A.zip,already have,8.9
3,graduates/ipeds_directory/HD2016.zip,already have,1.2
4,graduates/ipeds_completions/C2017_A.zip,already have,8.9
5,graduates/ipeds_directory/HD2017.zip,already have,1.1
6,graduates/ipeds_completions/C2018_A.zip,already have,8.5
7,graduates/ipeds_directory/HD2018.zip,already have,1.1
8,graduates/ipeds_completions/C2019_A.zip,already have,8.5
9,graduates/ipeds_directory/HD2019.zip,already have,1.0


### `outcomes`

In [8]:
log["outcomes"] = run_group("outcomes")
log["outcomes"]

,file,status,MB
0,outcomes/census_pseo/pseoe_ny.csv.gz,already have,0.9
1,outcomes/census_pseo/pseof_ny.csv.gz,already have,12.3
2,outcomes/scorecard_fos/Most-Recent-Cohorts-Fie...,already have,17.2
3,outcomes/acs_pums/csv_pny_2020-2024_5yr.zip,already have,140.1
4,outcomes/acs_pums/PUMS_Data_Dictionary_2020-20...,already have,0.3
5,outcomes/acs_pums/2024_ACS_Code_Lists.xlsx,already have,0.3
6,outcomes/acs_pums/2018-occupation-code-list-an...,already have,0.2
7,outcomes/acs_pums/2020_Census_Tract_to_2020_PU...,already have,1.7
8,outcomes/nyfed_recent_grads/college-labor-outc...,already have,0.0
9,outcomes/nyfed_recent_grads/College-labor-data...,already have,0.1


### `pay`

In [9]:
log["pay"] = run_group("pay")
log["pay"]

,file,status,MB
0,pay/dol_oflc_wages/OFLC_Wages_2021-22.zip,already have,12.2
1,pay/dol_oflc_wages/OFLC_Wages_2022-23.zip,already have,13.5
2,pay/dol_oflc_wages/OFLC_Wages_2023-24.zip,already have,13.7
3,pay/dol_oflc_wages/OFLC_Wages_2024-25.zip,already have,12.4
4,pay/dol_oflc_wages/OFLC_Wages_2025-26.zip,already have,12.3
5,pay/dol_oflc_wages/OFLC_Wages_2026-27.zip,already have,12.7
6,pay/nyc_payroll/k397-673e_fy2019_nonames.csv,already have,83.2
7,pay/nyc_payroll/k397-673e_fy2020_nonames.csv,already have,82.5
8,pay/nyc_payroll/k397-673e_fy2021_nonames.csv,already have,80.0
9,pay/nyc_payroll/k397-673e_fy2022_nonames.csv,already have,85.7


### `skills`

In [10]:
log["skills"] = run_group("skills")
log["skills"]

,file,status,MB
0,skills/onet/db_31_0_csv.zip,already have,16.2
1,skills/onet/db_30_0_text.zip,already have,13.5
2,skills/onet/2019_to_SOC_Crosswalk.xlsx,already have,0.4
3,skills/onet/Education_CIP_to_ONET_SOC.xlsx,already have,0.2


### `postings`

In [11]:
log["postings"] = run_group("postings")
log["postings"]

,file,status,MB
0,postings/nyc_jobs/kpav-sd4t_2026-10-12.csv,already have,18.1
1,postings/nyc_civil_titles/nzjr-3966_2026-10-12...,already have,0.4
2,postings/indeed_hiring_lab/aggregate_job_posti...,already have,0.2
3,postings/indeed_hiring_lab/state_job_postings_...,already have,2.6
4,postings/indeed_hiring_lab/metro_job_postings_...,already have,62.6
5,postings/indeed_hiring_lab/job_postings_by_sec...,already have,12.1


### `bls_timeseries` (BLS monthly series, bulk files)

Three monthly series from the BLS bulk-file server (`download.bls.gov/pub/time.series/`):
- CPI-U for the NY metro area (`cu`, area S12A). Used to show pay over time in today's dollars.
- **JOLTS for New York State** (`jt`, state 36). Job openings, hires, quits and layoffs. Industry detail is only published for the whole US.
- Unemployment rate (LAUS) for each of the 5 borough counties (`la`).

Why bulk files, not the API: the first run used the free BLS API, which without a key allows only a few requests per day per internet address (IP). Earlier research had used up the day's allowance, so all 3 pulls came back `REQUEST_NOT_PROCESSED`. The bulk files hold the full history (JOLTS from 2000, LAUS from 1976) with no daily cap, and they are the same numbers. BLS revises them in place, so they are dated snapshots. The series IDs we will read out of them are in `S.BLS_SERIES`.

In [12]:
log["bls_timeseries"] = run_group("bls_timeseries")
log["bls_timeseries"]

,file,status,MB
0,bls_timeseries/bls_jt/jt-data-1-AllItems_2026-...,already have,34.5
1,bls_timeseries/bls_jt/jt-series_2026-10-12.txt,already have,0.2
2,bls_timeseries/bls_jt/jt-state_2026-10-12.txt,already have,0.0
3,bls_timeseries/bls_jt/jt-industry_2026-10-12.txt,already have,0.0
4,bls_timeseries/bls_jt/jt-dataelement_2026-10-1...,already have,0.0
5,bls_timeseries/bls_jt/jt-ratelevel_2026-10-12.txt,already have,0.0
6,bls_timeseries/bls_jt/jt-sizeclass_2026-10-12.txt,already have,0.0
7,bls_timeseries/bls_jt/jt-area_2026-10-12.txt,already have,0.0
8,bls_timeseries/bls_jt/jt-period_2026-10-12.txt,already have,0.0
9,bls_timeseries/bls_jt/jt-footnote_2026-10-12.txt,already have,0.0


## Summary
Anything not `downloaded` or `already have` needs a look. A failed file is not retried silently; it shows up here.

In [13]:
summary = pd.concat(log, names=["group"]).reset_index(level=0)
print(summary["status"].value_counts().to_string())
problems = summary[~summary["status"].isin(["downloaded", "already have"])]
problems

status
already have    146
downloaded        9


,group,file,status,MB


In [14]:
manifest = pd.read_csv(S.MANIFEST)
print(f"manifest: {len(manifest)} files, {manifest['bytes'].sum()/1e9:.2f} GB")
manifest.groupby("group")["bytes"].agg(files="count", GB=lambda b: round(b.sum()/1e9, 2))

manifest: 155 files, 2.76 GB


,files,GB
group,,
bls_timeseries,25,0.07
graduates,27,0.11
hiring,5,0.29
jobs,26,1.35
licensing,39,0.00
outcomes,10,0.17
pay,13,0.64
postings,6,0.10
skills,4,0.03
